# Capítulo 9 · Laboratório 9b: Kaufman e as tartarugas
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap09b.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. A média adaptativa de Kaufman (KAMA) contra a média exponencial, o índice de eficiência como medida de ruído e as tartarugas simplificadas (Sistemas 1 e 2) em cinco mercados, com entradas sorteadas para testar a ideia de Eckhardt.

In [ ]:
!pip install -q yfinance

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
MERCADOS = {"Ibovespa": "^BVSP", "S&P 500": "^GSPC", "Dólar (R$)": "BRL=X", "Ouro": "GC=F", "Petróleo Brent": "BZ=F"}
FIM, CUSTO = "2026-10-01", 0.001
rng = np.random.default_rng(9)

CACHE = "dados_cap09b.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    dados, merc = pd.read_pickle(CACHE)
else:
    dados = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    merc = {k: yf.download(v, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, multi_level_index=False)
            for k, v in MERCADOS.items()}
    confere({"as 40 ações": dados, **merc})
    pd.to_pickle((dados, merc), CACHE)


def limpa(d, volume=True):
    d = d.dropna(subset=["Open", "High", "Low", "Close"])
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if volume:
        ok &= (d["Volume"] > 0) & (d["High"] > d["Low"])
    return d[ok]


def efficiency_ratio(c, n=10):
    """Índice de eficiência de Kaufman: |variação em n| ÷ soma das |variações de um dia|."""
    return (c - c.shift(n)).abs() / c.diff().abs().rolling(n).sum()


def kama(c, n=10, rapido=2, lento=30):
    """KAMA (Kaufman, Trading Systems and Methods, 5ª ed., p. 780 a 782)."""
    er = efficiency_ratio(c, n).to_numpy()
    f, s = 2 / (rapido + 1), 2 / (lento + 1)
    sc = (er * (f - s) + s) ** 2
    C = c.to_numpy()
    out = np.full(len(C), np.nan)
    out[n] = C[n]
    for i in range(n + 1, len(C)):
        out[i] = out[i - 1] + sc[i] * (C[i] - out[i - 1])
    return pd.Series(out, index=c.index)


def resultado(pos, c):
    """Comprado (1) ou fora (0) com o sinal de ontem; custo de 0,10% por ida e volta."""
    r = c.pct_change().fillna(0)
    p = pos.shift(1).fillna(0)
    liq = p * r - p.diff().abs().fillna(0) * CUSTO / 2
    eq = (1 + liq).cumprod()
    anos = len(c) / 252
    return {"ao ano": eq.iloc[-1] ** (1 / anos) - 1, "pior queda": (eq / eq.cummax() - 1).min(),
            "operações por ano": p.diff().clip(lower=0).sum() / anos, "tempo comprado": p.mean(),
            "Sharpe": np.sqrt(252) * liq.mean() / liq.std()}

## Parte 1: KAMA x MME

In [ ]:
ib = limpa(merc["Ibovespa"], volume=False)["Close"].loc["2006":]
k = kama(ib)
filtro = 0.1 * k.diff().rolling(20).std()                     # Kaufman: filtro pequeno para mudança de direção
estado, p = [], 0.0
for dk, fl in zip(k.diff().to_numpy(), filtro.to_numpy()):
    if not np.isnan(dk) and not np.isnan(fl):
        if dk > fl:
            p = 1.0
        elif dk < -fl:
            p = 0.0
    estado.append(p)
regras = {"KAMA(10, 2, 30) subindo": pd.Series(estado, index=ib.index),
          "MME de 20 subindo": (ib.ewm(span=20, adjust=False).mean().diff() > 0).astype(float),   # o mesmo que preço acima da MME
          "preço acima da MM de 200": (ib > ib.rolling(200).mean()).astype(float),
          "comprar e segurar": pd.Series(1.0, index=ib.index)}
print("Parte 1. Ibovespa 2006 a 2026, comprado ou fora (custo de 0,10% por ida e volta)")
for nome, pos in regras.items():
    x = resultado(pos, ib)
    print(f"   {nome:<28} {x['ao ano']:+.1%} ao ano | pior queda {x['pior queda']:.0%} | {x['operações por ano']:.1f} operações por ano"
          f" | comprado {x['tempo comprado']:.0%} do tempo | Sharpe {x['Sharpe']:.2f}")
tab = []
for t in TICKERS:
    c = limpa(dados[t])["Close"].loc["2006":]
    kk = kama(c)
    fl = 0.1 * kk.diff().rolling(20).std()
    est, p = [], 0.0
    for dk, f_ in zip(kk.diff().to_numpy(), fl.to_numpy()):
        if not np.isnan(dk) and not np.isnan(f_):
            p = 1.0 if dk > f_ else (0.0 if dk < -f_ else p)
        est.append(p)
    tab.append({"KAMA": resultado(pd.Series(est, index=c.index), c)["Sharpe"],
                "MME 20": resultado((c.ewm(span=20, adjust=False).mean().diff() > 0).astype(float), c)["Sharpe"],
                "segurar": resultado(pd.Series(1.0, index=c.index), c)["Sharpe"],
                "op_kama": resultado(pd.Series(est, index=c.index), c)["operações por ano"],
                "op_mme": resultado((c.ewm(span=20, adjust=False).mean().diff() > 0).astype(float), c)["operações por ano"]})
T1 = pd.DataFrame(tab, index=TICKERS)
print(f"   40 ações (Sharpe mediano): KAMA {T1['KAMA'].median():.2f} com {T1['op_kama'].median():.0f} operações por ano |"
      f" MME de 20 subindo {T1['MME 20'].median():.2f} com {T1['op_mme'].median():.0f} | comprar e segurar {T1['segurar'].median():.2f};"
      f" KAMA melhor que a MME em {np.mean(T1['KAMA'] > T1['MME 20']):.0%} das ações")

## Parte 2: ruído x tendência

In [ ]:
lin = []
for t in TICKERS:
    d = limpa(dados[t]).loc["2006":]
    c = d["Close"]
    er = efficiency_ratio(c, 20).median()
    alta, baixa = d["High"].rolling(20).max().shift(1), d["Low"].rolling(10).min().shift(1)
    est, p = [], 0.0
    for ci, a, b in zip(c.to_numpy(), alta.to_numpy(), baixa.to_numpy()):
        if p == 0 and ci > a:
            p = 1.0
        elif p == 1 and ci < b:
            p = 0.0
        est.append(p)
    x = resultado(pd.Series(est, index=c.index), c)
    lin.append({"ativo": t.replace(".SA", ""), "ER": er, "Sharpe Donchian": x["Sharpe"],
                "a mais que segurar": x["Sharpe"] - resultado(pd.Series(1.0, index=c.index), c)["Sharpe"]})
T2 = pd.DataFrame(lin).set_index("ativo")
print(f"\nParte 2. Índice de eficiência de 20 dias (mediana de 2006 a 2026) e o Donchian 20/10 só comprado, 40 ações")
print(f"   ER de {T2['ER'].min():.2f} a {T2['ER'].max():.2f}; correlação de postos entre ER e Sharpe do Donchian:"
      f" {T2['ER'].rank().corr(T2['Sharpe Donchian'].rank()):+.2f}; entre ER e a vantagem sobre segurar:"
      f" {T2['ER'].rank().corr(T2['a mais que segurar'].rank()):+.2f}")
for nome in MERCADOS:
    print(f"   {nome:<15} ER mediano {efficiency_ratio(limpa(merc[nome], volume=False)['Close'].loc['2006':], 20).median():.2f}")

## Parte 3: as tartarugas

In [ ]:
def preparar(d):
    pc = d["Close"].shift()
    tr = pd.concat([d["High"] - d["Low"], (d["High"] - pc).abs(), (d["Low"] - pc).abs()], axis=1).max(axis=1)
    out = d.copy()
    out["N"] = tr.rolling(20).mean()                           # Covel, p. 80: média de 20 dias do true range
    for n in (10, 20, 55):
        out[f"max{n}"] = d["High"].rolling(n).max().shift(1)
        out[f"min{n}"] = d["Low"].rolling(n).min().shift(1)
    return out.dropna()


def operacoes(d, ent, sai, dias_entrada=None, direcao=None):
    """Uma posição por vez. Entra no toque do canal de ent dias (ou nos dias sorteados), sai no toque do canal
    de sai dias ou no stop de 2N. Devolve as operações e o resultado diário em % do capital (1% por N)."""
    O, H, L, C, N = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close", "N"))
    mx, mn = d[f"max{ent}"].to_numpy(), d[f"min{ent}"].to_numpy()
    sx, sn = d[f"max{sai}"].to_numpy(), d[f"min{sai}"].to_numpy()
    diario = np.zeros(len(C))
    ops, i = [], 1
    sorteados = set(dias_entrada) if dias_entrada is not None else None
    while i < len(C) - 1:
        lado = 0
        if sorteados is None:
            if H[i] > mx[i]:
                lado, e = 1, max(mx[i], O[i])
            elif L[i] < mn[i]:
                lado, e = -1, min(mn[i], O[i])
        elif i in sorteados:
            lado = direcao[i]
            e = O[i]
        if lado == 0:
            i += 1
            continue
        n_ = N[i - 1]
        stop = e - 2 * n_ * lado
        f = 0.01 / n_                                        # 1% do capital por N
        diario[i] += f * lado * (C[i] - e) - 0.01 * CUSTO / 2 * e / n_
        j, saida = i + 1, None
        while j < len(C):
            if lado == 1:
                nivel = max(stop, sn[j])
                if L[j] <= nivel:
                    saida = min(nivel, O[j])
            else:
                nivel = min(stop, sx[j])
                if H[j] >= nivel:
                    saida = max(nivel, O[j])
            if saida is not None:
                diario[j] += f * lado * (saida - C[j - 1]) - 0.01 * CUSTO / 2 * saida / n_
                break
            diario[j] += f * lado * (C[j] - C[j - 1])
            j += 1
        if saida is None:
            break
        ops.append({"entrada": d.index[i], "lado": lado, "R": lado * (saida - e) / (2 * n_)})
        i = j + 1
    return pd.DataFrame(ops), pd.Series(diario, index=d.index)


def medidas(diario):
    eq = (1 + diario).cumprod()
    anos = len(diario) / 252
    return eq.iloc[-1] ** (1 / anos) - 1, (eq / eq.cummax() - 1).min(), np.sqrt(252) * diario.mean() / diario.std()


PREP = {k: preparar(limpa(v, volume=False).loc["2005":]) for k, v in merc.items()}
S1t, S1f, S2, alvo = {}, {}, {}, {}
for nome, d in PREP.items():
    o1, d1 = operacoes(d, 20, 10)
    perdeu_antes = np.r_[True, (o1["R"].to_numpy() <= 0)[:-1]]     # filtro do S1: só entra se a anterior (teórica) perdeu
    o1f = o1[perdeu_antes]
    dias_ok = set(d.index.get_indexer(o1f["entrada"]))
    lados = pd.Series(0, index=range(len(d)))
    lados.iloc[list(dias_ok)] = o1f["lado"].to_numpy()
    o1b, d1f = operacoes(d, 20, 10, dias_entrada=dias_ok, direcao=lados.to_numpy())
    o2, d2 = operacoes(d, 55, 20)
    S1t[nome], S1f[nome], S2[nome] = (o1, d1), (o1b, d1f), (o2, d2)
print("\nParte 3. Tartarugas simplificadas, 2005 a 2026 (comprado e vendido; 1% do capital por N; stop de 2N; sem piramidação)")
print("   mercado         | S1 sem filtro: op. acerto R médio | S1 com filtro: op. R médio | S2 55/20: op. acerto R médio | % do lucro do S2 nas 10% melhores")
for nome in PREP:
    a, b, c_ = S1t[nome][0], S1f[nome][0], S2[nome][0]
    melhores = c_["R"].nlargest(max(1, len(c_) // 10)).sum() / c_.loc[c_["R"] > 0, "R"].sum()
    print(f"   {nome:<15} | {len(a):4d} {np.mean(a['R'] > 0):5.0%} {a['R'].mean():+.2f} | {len(b):4d} {b['R'].mean():+.2f}"
          f" | {len(c_):4d} {np.mean(c_['R'] > 0):5.0%} {c_['R'].mean():+.2f} | {melhores:.0%}")
cart = {}
for rot, base in (("S1 com filtro", S1f), ("S2 (55/20)", S2)):
    cart[rot] = pd.concat([base[n][1] for n in PREP], axis=1, sort=True).fillna(0).sum(axis=1)
cart["S1 e S2 juntos"] = cart["S1 com filtro"] + cart["S2 (55/20)"]
print("   carteira dos cinco mercados (soma dos resultados diários):")
for rot, s in cart.items():
    a_, q_, sh_ = medidas(s.loc["2006":])
    print(f"      {rot:<15} {a_:+.1%} ao ano | pior queda {q_:.0%} | Sharpe {sh_:.2f}")
for nome in ("Ibovespa", "Ouro"):
    a_, q_, _ = medidas(PREP[nome]["Close"].pct_change().fillna(0).loc["2006":])
    print(f"      comprar e segurar {nome}: {a_:+.1%} ao ano | pior queda {q_:.0%}")

# Eckhardt: entradas sorteadas com a mesma saída do S2 e o mesmo tamanho
reais = np.mean(np.concatenate([S2[n][0]["R"].to_numpy() for n in PREP]))
sorteio = []
for _ in range(200):
    rs = []
    for nome, d in PREP.items():
        n_op = len(S2[nome][0])
        dias = rng.choice(np.arange(2, len(d) - 30), size=n_op * 4, replace=False)
        lado = np.zeros(len(d), int)
        lado[dias] = rng.choice([-1, 1], size=len(dias))
        o, _ = operacoes(d, 55, 20, dias_entrada=set(dias), direcao=lado)
        rs.append(o["R"].to_numpy()[:n_op])
    sorteio.append(np.mean(np.concatenate(rs)))
sorteio = np.array(sorteio)
print(f"   entradas sorteadas (lado ao acaso) com a saída do S2 e stop de 2N, 200 sorteios: R médio {sorteio.mean():+.3f}"
      f" (5% a 95%: {np.percentile(sorteio, 5):+.3f} a {np.percentile(sorteio, 95):+.3f}); S2 de verdade {reais:+.3f};"
      f" o S2 bateu {np.mean(sorteio < reais):.0%} dos sorteios")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `efficiency_ratio` e `kama` da célula de abertura. Mostre onde o índice de eficiência entra na constante `sc`, por que `sc` é elevada ao quadrado e por que, com rápida de 2 e lenta de 30, a KAMA anda como uma média exponencial de uns 3,5 períodos quando o índice vale 1 e de uns 480 quando vale perto de zero." Confira com a Parte 1, que imprime a linha da KAMA (10, 2, 30) da Tabela 9.2: +4,5% ao ano, 8,7 operações por ano e Sharpe de 0,36 no Ibovespa.
2. **Adaptar:** "Crie uma célula nova no fim do notebook que baixe [ATIVO], com o ticker do Yahoo Finance, nos mesmos moldes do dicionário `merc` da célula de abertura (desde 2005, preços ajustados), e rode nele as quatro regras da Parte 1 com as funções `kama` e `resultado`. Não mude o dicionário `MERCADOS`, que muda a carteira da Parte 3, nem a fórmula da KAMA, os parâmetros (10, 2, 30) ou o custo de 0,10%." Antes de olhar o ativo novo, confira se a Parte 1 continua dando os números do Ibovespa da Tabela 9.2.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 1, troque a KAMA (10, 2, 30) pela (20, 2, 30). Menos operações? Mais retorno?
2. Na Parte 3, mude o risco de 1% por N para 0,5%. O que acontece com a pior queda da carteira?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.